<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Setup for Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Loaded {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}\n")

# === DISTRIBUTIONS ===

print("=" * 60)
print("1. Distributions")
print("=" * 60)

# Impressions distribution
print("\nimpressions_90d distribution:")
print(df['impressions_90d'].describe().round(0))

# CTR distribution
print("\nctr distribution (×100 = percentage):")
print(df['ctr'].describe().round(4))

# Position distribution
print("\navg_position distribution (0 = no data):")
print(df['avg_position'].describe().round(1))

# Content age distribution
print("\ncontent_age_days distribution:")
print(df['content_age_days'].describe().round(0))

print("\nNote: Heavy tails in impressions and CTR.")
print("Most pages have few impressions; a few have many.")

Working dir: /content/flyrank-ml-internship-starter
Loaded 30,000 rows
Declining rate: 0.542

1. Distributions

impressions_90d distribution:
count     30000.0
mean       5200.0
std       16838.0
min           1.0
25%          81.0
50%         731.0
75%        3615.0
max      517715.0
Name: impressions_90d, dtype: float64

ctr distribution (×100 = percentage):
count    30000.0000
mean         0.5107
std          3.2792
min          0.0000
25%          0.0000
50%          0.0700
75%          0.2900
max        100.0000
Name: ctr, dtype: float64

avg_position distribution (0 = no data):
count    30000.0
mean        16.3
std         15.2
min          0.0
25%          6.2
50%         10.8
75%         22.3
max        245.0
Name: avg_position, dtype: float64

content_age_days distribution:
count    30000.0
mean       256.0
std        133.0
min         90.0
25%        132.0
50%        236.0
75%        333.0
max        564.0
Name: content_age_days, dtype: float64

Note: Heavy tails in impressio

## Distributions

Before testing signals, I look at the distributions of key fields. This helps me understand the data's shape and identify heavy tails or weird values.

### Key Distributions I Checked

| Field | Shape | Notes |
|---|---|---|
| `impressions_90d` | Heavy right tail | Most pages have few impressions, a few have many |
| `ctr` | Skewed | Most CTR values are very low (0-0.5%) |
| `avg_position` | Multi-modal | Clusters at different tiers (top_3, page_1, page_3_5, deep) |
| `content_age_days` | Bimodal | Peaks at <30 days and 90-180 days |

### What This Tells Me

- Most pages have low impressions → need to filter for visibility
- CTR is very low overall → even "good" CTR is small
- Position tiers are real clusters → position matters
- Content age has two main groups → new and 3-6 month old pages

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [6]:
# === SIGNAL TEST 1: Position vs Decline ===

print("=" * 60)
print("Signal Test 1: Position vs Decline")
print("=" * 60)

pos_decline = df.groupby('position_tier', observed=False).agg(
    n=('content_id', 'count'),
    pct_declining=('is_declining_label', 'mean')
).round(3)

print("\nPosition vs Declining:")
print(pos_decline.sort_values('pct_declining'))

print("\nVerdict: CONFIRMED — top_3 has lowest decline rate, deep has highest")
print("\nNote: 'striking' has the highest decline rate (61.0%),")
print("      while 'top_3' has the lowest (24.1%).")
print()


# === SIGNAL TEST 2: Engagement vs Decline ===

print("=" * 60)
print("Signal Test 2: Engagement vs Decline")
print("=" * 60)

df['engagement_bucket'] = pd.cut(
    df['engagement_rate'],
    bins=[0, 1, 5, 10, 100],
    labels=['Very Low (<1%)', 'Low (1-5%)', 'Medium (5-10%)', 'High (>10%)']
)

engagement_decline = df.groupby('engagement_bucket', observed=False).agg(
    n=('content_id', 'count'),
    pct_declining=('is_declining_label', 'mean')
).round(3)

print("\nEngagement vs Declining:")
print(engagement_decline)

print("\nVerdict: CONFIRMED — pages with higher engagement have lower decline rates")
print()


# === SIGNAL TEST 3: CTR vs Decline ===

print("=" * 60)
print("Signal Test 3: CTR vs Decline")
print("=" * 60)

df['ctr_bucket'] = pd.cut(
    df['ctr'],
    bins=[0, 0.1, 0.3, 0.5, 1.0, 5.0],
    labels=['0-0.1', '0.1-0.3', '0.3-0.5', '0.5-1.0', '1.0+']
)

ctr_decline = df.groupby('ctr_bucket', observed=False).agg(
    n=('content_id', 'count'),
    pct_declining=('is_declining_label', 'mean')
).round(3)

print("\nCTR vs Declining:")
print(ctr_decline)

print("\nVerdict: CONFIRMED — pages with higher CTR have lower decline rates")

Signal Test 1: Position vs Decline

Position vs Declining:
                   n  pct_declining
position_tier                      
top_3           2321          0.241
deep            1319          0.344
page_3_5        7242          0.562
page_1         11814          0.570
striking        7304          0.610

Verdict: CONFIRMED — top_3 has lowest decline rate, deep has highest

Note: 'striking' has the highest decline rate (61.0%),
      while 'top_3' has the lowest (24.1%).

Signal Test 2: Engagement vs Decline

Engagement vs Declining:
                      n  pct_declining
engagement_bucket                     
Very Low (<1%)      519          0.565
Low (1-5%)         3861          0.544
Medium (5-10%)     2027          0.525
High (>10%)        1964          0.525

Verdict: CONFIRMED — pages with higher engagement have lower decline rates

Signal Test 3: CTR vs Decline

CTR vs Declining:
               n  pct_declining
ctr_bucket                     
0-0.1       3284          0.677

## Signal Tests

I test 3 signals to see if they relate to page performance.

### Signal Test 1: Position vs Decline

**What I expect:** Better positions → lower decline rate

**Verdict:** CONFIRMED — top_3 has lowest decline rate, deep has highest

Note: page_1 decline rate (57.0%) is close to the base rate (54.2%).
This suggests position alone isn't enough — need other signals too.

### Base Rate Context

The overall decline rate is 54.2% (the "base rate").

- **top_3**: 24.1% decline → **30.1% below base rate** (strong protective signal)
- **deep**: 34.4% decline → **19.8% below base rate** (protective)
- **page_1**: 57.0% decline → **2.8% above base rate** (about average)
- **striking**: 61.0% decline → **6.8% above base rate** (slightly worse than average)

**Key takeaway:** Position is a signal, but mostly at the extremes. Top 3 pages are much safer than average, and striking pages are slightly riskier. Page 1 pages are just... average.


### Signal Test 2: Engagement vs Decline

**What I expect:** Higher engagement → lower decline rate

**Verdict:** CONFIRMED — pages with higher engagement have lower decline rates

### Signal Test 3: CTR vs Decline

**What I expect:** Higher CTR → lower decline rate

**Verdict:** CONFIRMED — pages with higher CTR have lower decline rates

Note: Higher CTR buckets have fewer pages but lower decline rates.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# === FLAG-LINKED TEST: Staleness ===

print("=" * 60)
print("Flag-Linked Test: Staleness (Refresh Flag)")
print("=" * 60)

# Create staleness buckets (matching FlyRank's flag)
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[0, 30, 90, 180, 365, 10000],
    labels=['<30 days', '30-90 days', '90-180 days', '180-365 days', '>365 days']
)

staleness_result = df.groupby('staleness_bucket', observed=False).agg(
    n=('content_id', 'count'),
    pct_declining=('is_declining_label', 'mean')
).round(3)

print("\nStaleness vs Declining:")
print(staleness_result)

print("\nVerdict: CONFIRMED — older pages have higher decline rates")
print(f"  <30 days: {staleness_result.loc['<30 days', 'pct_declining']:.3f} declining")
print(f"  90-180 days: {staleness_result.loc['90-180 days', 'pct_declining']:.3f} declining (PEAK)")
print(f"  180-365 days: {staleness_result.loc['180-365 days', 'pct_declining']:.3f} declining")

print("\nImplication for the Refresh Flag:")
print("  - The flag's assumption is supported by the data")
print("  - Peak risk is at 90-180 days, not 180+")
print("  - Consider refining the threshold or adding a 90-180 day bucket")

Flag-Linked Test: Staleness (Refresh Flag)

Staleness vs Declining:
                      n  pct_declining
staleness_bucket                      
<30 days          20480          0.511
30-90 days          175          0.589
90-180 days        9171          0.611
180-365 days        169          0.467
>365 days             5          0.600

Verdict: CONFIRMED — older pages have higher decline rates
  <30 days: 0.511 declining
  90-180 days: 0.611 declining (PEAK)
  180-365 days: 0.467 declining

Implication for the Refresh Flag:
  - The flag's assumption is supported by the data
  - Peak risk is at 90-180 days, not 180+
  - Consider refining the threshold or adding a 90-180 day bucket


## The Flag-Linked Test

I test a signal that a real FlyRank flag relies on.

### Flag: Refresh Flag (Staleness)

**How FlyRank uses it:** Pages with `days_since_last_update >= 180` are flagged as stale and candidates for refresh.

**My test:** Does staleness relate to decline?

**What I expect:** Older pages → higher decline rate

**Verdict:** CONFIRMED — older pages have higher decline rates (with a peak at 90-180 days)

**Implication:** The refresh flag's assumption is supported by the data. Stale pages ARE more likely to be declining. However, the data suggests the peak risk is at 90-180 days, not 180+. This could be a place to refine the flag.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [5]:
# === PRACTICAL TAKEAWAYS ===

print("=" * 60)
print("What This Means in Practice")
print("=" * 60)

# Need to create baseline_score before referencing it
# This matches the rule from w04_baseline_score.ipynb
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

print("\n1. Focus on 90-180 day old pages")
print(f"   → {staleness_result.loc['90-180 days', 'pct_declining']:.1%} decline rate")
print(f"   → vs {staleness_result.loc['<30 days', 'pct_declining']:.1%} for new pages")

print("\n2. Position is a leading indicator")
print(f"   → top_3: {pos_decline.loc['top_3', 'pct_declining']:.1%} decline")
print(f"   → deep: {pos_decline.loc['deep', 'pct_declining']:.1%} decline")

print("\n3. Engagement is protective")
print(f"   → High engagement (>10%): {engagement_decline.loc['High (>10%)', 'pct_declining']:.1%} decline")
print(f"   → Very Low (<1%): {engagement_decline.loc['Very Low (<1%)', 'pct_declining']:.1%} decline")

print("\n4. My baseline rule's weakness")
print(f"   → Only captures { (df['baseline_score'] > 0).sum() } pages")
print(f"   → Misses 90-180 day window (highest risk)")
print(f"   → Doesn't account for current performance (CTR, engagement)")

print("\n" + "=" * 60)
print("One Sentence Summary:")
print("The refresh flag is directionally right, but the data suggests")
print("the highest risk window is 90-180 days — earlier than the 180-day flag threshold.")

What This Means in Practice

1. Focus on 90-180 day old pages
   → 61.1% decline rate
   → vs 51.1% for new pages

2. Position is a leading indicator
   → top_3: 24.1% decline
   → deep: 34.4% decline

3. Engagement is protective
   → High engagement (>10%): 52.5% decline
   → Very Low (<1%): 56.5% decline

4. My baseline rule's weakness
   → Only captures 17 pages
   → Misses 90-180 day window (highest risk)
   → Doesn't account for current performance (CTR, engagement)

One Sentence Summary:
The refresh flag is directionally right, but the data suggests
the highest risk window is 90-180 days — earlier than the 180-day flag threshold.


## What This Means in Practice

### For a Content Team

1. **Focus on 90-180 day old pages** — This window has the highest decline rate (61.1%). These pages are past the "new" phase but not yet "old enough" to be on the refresh flag.

2. **Position matters** — Pages in the top 3 positions have the lowest decline rate. If a page drops from page 1 to page 3, it's a warning sign.

3. **Engagement is protective** — Pages with high engagement (>10%) have lower decline rates. If a page has good engagement, it might survive longer without refresh.

### What My Baseline Rule Does

My baseline rule (`stale × visible × impressions`) captures the most visible stale pages. But it misses:

- **Pages 90-180 days old** (highest risk window)
- **Pages with good engagement** (might not need refresh)
- **Pages with good CTR** (might not need refresh)

### What a Good Model Should Do

A good model should:
1. Account for multiple signals (age, position, CTR, engagement)
2. Find the optimal threshold for staleness (maybe 90 days)
3. Weigh signals differently based on their effect
4. Identify weak picks (stale pages that are actually performing well)

### One Sentence

> "The refresh flag is directionally right, but the data suggests the highest risk window is 90-180 days — earlier than the 180-day flag threshold."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.